# Notebook 02 -- Filtering Pipeline

> **Note:** This notebook is pre-run. The 1,488 original raw parquet files were deleted after filtering to save disk space. Row counts before filtering are taken from the processing log (`02_data_cleaning/logs/data_profile.md`). The current working files at `C:\BA_Data\` reflect the post-filtering state.

---

This notebook documents the three filtering steps applied to the raw DSA data before analysis. Each step narrows the dataset to the records relevant for this thesis.

In [5]:
from pathlib import Path

import polars as pl

DATA_ROOT = Path(r"C:\BA_Data")
TIKTOK_FILE = DATA_ROOT / "tiktok_de_2025.parquet"
X_FILE = DATA_ROOT / "x_de_2025.parquet"

## Step 0 -- Column Selection

The raw files contain **37 columns**. The first filtering decision was to restrict the working dataset to the 9 thesis-relevant columns, plus 1 derived column added during this step.

*(Decision reference: CD-20260426-02 in `02_data_cleaning/docs/cleaning_decisions.md`)*

| | Columns |
|---|---|
| **Selected from raw (9)** | `uuid`, `category`, `content_type`, `automated_detection`, `automated_decision`, `territorial_scope`, `application_date`, `platform_name`, `decision_visibility` |
| **Dropped (27)** | `decision_facts` (free text), `illegal_content_explanation` (free text), `decision_monetary`, `decision_provision`, `decision_account`, `account_type`, `source_identity`, `category_specification`, `content_language`, `platform_uid`, `created_at`, and 16 others outside thesis scope |

**Rationale:** The retained columns directly support the core analytical dimensions -- unique records, harm category, content modality, automation indicators, temporal filtering, platform comparison, territorial scope, and visibility outcome. Dropped columns are either free-text (not quantifiable), predominantly empty, or outside the research questions.

In [6]:
# TikTok -- first 5 rows after column selection
print("TikTok:")
display(pl.scan_parquet(str(TIKTOK_FILE)).head(5).collect())

# X -- first 5 rows after column selection
print("X:")
display(pl.scan_parquet(str(X_FILE)).head(5).collect())

TikTok:


uuid,category,content_type,automated_detection,automated_decision,territorial_scope,application_date,platform_name,decision_visibility,api_version
str,str,str,str,str,str,datetime[ms],str,str,str
"""0b3c3647-9806-419c-bd99-de66b6…","""STATEMENT_CATEGORY_SCOPE_OF_PL…","""[""CONTENT_TYPE_VIDEO""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2025-01-01 00:00:00,"""TikTok""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""8415527b-855e-437b-a698-4eef9b…","""STATEMENT_CATEGORY_SCOPE_OF_PL…","""[""CONTENT_TYPE_VIDEO""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2025-01-01 00:00:00,"""TikTok""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""27ebdca3-085f-47d3-91ad-79e35b…","""STATEMENT_CATEGORY_ILLEGAL_OR_…","""[""CONTENT_TYPE_TEXT""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2025-01-01 00:00:00,"""TikTok""","""[""DECISION_VISIBILITY_CONTENT_…","""v1"""
"""73aa4ad5-3480-4597-b1aa-c3beb2…","""STATEMENT_CATEGORY_SCOPE_OF_PL…","""[""CONTENT_TYPE_VIDEO""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2025-01-01 00:00:00,"""TikTok""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""f277ba47-b2ab-4e97-b5a8-ec7e41…","""STATEMENT_CATEGORY_SCOPE_OF_PL…","""[""CONTENT_TYPE_VIDEO""]""","""Yes""","""AUTOMATED_DECISION_FULLY""","""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2025-01-01 00:00:00,"""TikTok""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""


X:


uuid,category,content_type,automated_detection,automated_decision,territorial_scope,application_date,platform_name,decision_visibility,api_version
str,str,str,str,str,str,datetime[ms],str,str,str
"""457e68f1-ed59-470e-9a6b-9902d2…","""STATEMENT_CATEGORY_SCAMS_AND_F…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""",2025-01-01 00:00:00,"""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""a07aa67e-adbe-4f3e-ab4b-c485b4…","""STATEMENT_CATEGORY_SCAMS_AND_F…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""",2025-01-01 00:00:00,"""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""604e6f72-6a67-43ab-8f77-003c66…","""STATEMENT_CATEGORY_SCAMS_AND_F…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""",2025-01-01 00:00:00,"""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""9487ad6c-e72d-493d-b25f-4f9826…","""STATEMENT_CATEGORY_PORNOGRAPHY…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""",2025-01-01 00:00:00,"""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""
"""3eadd52f-deef-45c9-9f3d-d419a2…","""STATEMENT_CATEGORY_UNSAFE_AND_…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…","""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""[""DE""]""",2025-01-01 00:00:00,"""X""","""[""DECISION_VISIBILITY_OTHER""]""","""v1"""


## Step 1 -- Territorial Scope Filter (DE)

The DSA Transparency Database contains SoRs for all EEA countries. This thesis focuses on **Germany (DE)** only.

The filter kept any record where the `territorial_scope` field contains `DE` -- this includes both single-country (`DE`) and multi-country strings (`DE,FR,NL,IT,...`). Multi-country records represent moderation decisions that apply EEA-wide and happen to cover Germany.

**Script:** `02_data_cleaning/scripts/filter_split_save_de_polars.py`

| Platform | Rows before DE filter | Rows after DE filter | Retention |
|---|---|---|---|
| TikTok | 1,002,729,268 | 999,079,277 | 99.6% |
| X | 670,093 | 183,324 | 27.4% |

**Key finding:** TikTok retained almost all records because it enforces content moderation EEA-wide (most records include DE). X retained only 27.4% because it enforces country-by-country -- the majority of its records target other EEA countries.

In [7]:
# Show what territorial_scope values look like after the DE filter
# TikTok: multi-country strings containing DE
print("TikTok -- sample territorial_scope values:")
(
    pl.scan_parquet(str(TIKTOK_FILE))
    .select("territorial_scope")
    .head(8)
    .collect()
    .to_series()
    .to_list()
)


TikTok -- sample territorial_scope values:


['["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","LU","LV","MT","NL","NO","PL","PT","RO","SE","SI","SK"]',
 '["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","HU","IE","IT","LI","LT","

In [8]:
# X: single country code
print("X -- unique territorial_scope values:")
(
    pl.scan_parquet(str(X_FILE))
    .select("territorial_scope")
    .unique()
    .collect()
    .to_series()
    .to_list()
)

X -- unique territorial_scope values:


['["DE"]']

## Step 2 -- Date Range Filter (2025)

Some records in the DE-filtered files had `application_date` values outside the 2025 calendar year. These were dropped to keep the dataset within the thesis scope.

**Script:** `02_data_cleaning/scripts/apply_date_filter.py`

| Platform | Rows before date filter | Rows dropped | Rows after date filter |
|---|---|---|---|
| TikTok | 999,079,277 | 1,391,010 | 997,688,267 |
| X | 183,324 | 3 | 183,321 |

In [9]:
# Confirm the date range of the current working files
for label, file in [("TikTok", TIKTOK_FILE), ("X", X_FILE)]:
    result = (
        pl.scan_parquet(str(file))
        .select(
            pl.col("application_date").min().alias("earliest"),
            pl.col("application_date").max().alias("latest"),
        )
        .collect()
    )
    print(f"{label}: {result['earliest'][0]} to {result['latest'][0]}")

TikTok: 2025-01-01 00:00:00 to 2025-12-12 00:00:00
X: 2025-01-01 00:00:00 to 2025-12-31 00:00:00


## Step 2b -- Adding `api_version` Column

| | Columns |
|---|---|
| **Derived (1)** | `api_version` -- added based on whether `application_date` falls before or after July 1, 2025 |

In [10]:
# TikTok -- first 5 rows showing the derived api_version column
pl.scan_parquet(str(TIKTOK_FILE)).select(["application_date", "api_version"]).head(5).collect()

application_date,api_version
datetime[ms],str
2025-01-01 00:00:00,"""v1"""
2025-01-01 00:00:00,"""v1"""
2025-01-01 00:00:00,"""v1"""
2025-01-01 00:00:00,"""v1"""
2025-01-01 00:00:00,"""v1"""


## Step 3 -- API Version Split

On **July 1, 2025** the DSA Commission released API v2, which changed the `category` vocabulary. Records submitted before this date used v1 labels; records from July 1 onwards use v2 labels. The `api_version` column captures this distinction.

This split matters because v1 and v2 category labels are not directly comparable -- categories were renamed, removed, and introduced. Harmonization is required before any cross-period analysis. See Notebook 03.

In [11]:
# API version distribution per platform
for label, file in [("TikTok", TIKTOK_FILE), ("X", X_FILE)]:
    dist = (
        pl.scan_parquet(str(file))
        .group_by("api_version")
        .agg(pl.len().alias("count"))
        .sort("api_version")
        .collect()
    )
    total = dist["count"].sum()
    print(f"\n{label} ({total:,} total):")
    for row in dist.iter_rows(named=True):
        pct = row["count"] / total * 100
        print(f"  {row['api_version']}: {row['count']:>13,}  ({pct:.1f}%)")


TikTok (997,688,267 total):
  v1:   830,222,338  (83.2%)
  v2:   167,465,929  (16.8%)

X (183,321 total):
  v1:       132,191  (72.1%)
  v2:        51,130  (27.9%)


## Summary -- Filtering Results

| Step | TikTok rows | X rows |
|---|---|---|
| Raw (before any filtering) | 1,002,729,268 | 670,093 |
| After DE territorial filter | 999,079,277 | 183,324 |
| After 2025 date filter | 997,688,267 | 183,321 |
| -- of which v1 | 830,222,338 | 132,191 |
| -- of which v2 | 167,465,929 | 51,130 |

